# Testing Bio Portal's RESTFUL API

https://data.bioontology.org/documentation#nav_resource_endpoints


"""
What is a .owl file?
An OWL file (Web Ontology Language) is an XML-based file format used to represent ontologies - formal descriptions of knowledge domains. 
"""

In [ ]:
"""
classes_search_terms.txt

heart
lung
experiment
human
brain
melanoma

"""



In [ ]:

# CLASS SEARCH
# https://github.com/ncbo/ncbo_rest_sample_code/blob/master/python/python3/classes_search.py

import urllib.request, urllib.error, urllib.parse
import json
import os
from pprint import pprint
from dotenv import load_dotenv

REST_URL = "http://data.bioontology.org"

# Create a .env file with your API key
# Format: BIO_PORTAL_API_KEY=your_api_key_here
load_dotenv()
API_KEY = os.environ.get("BIO_PORTAL_API_KEY", "")


def get_json(url):
    opener = urllib.request.build_opener()
    opener.addheaders = [('Authorization', 'apikey token=' + API_KEY)]
    return json.loads(opener.open(url).read())

# Get list of search terms
# path = os.path.join(os.path.dirname(__file__), 'classes_search_terms.txt')
# terms_file = open(path, "r")
# terms = []
# for line in terms_file:
#     terms.append(line)
terms = ['heart']

# Do a search for every term
search_results = []
for term in terms:
    search_results.append(get_json(REST_URL + "/search?q=" + term)["collection"])

# Print the results
for result in search_results:
    pprint(result)


In [ ]:
# Explore the DOID (Human Disease Ontology) hierarchy

import urllib.request, urllib.error, urllib.parse
import json
from pprint import pprint
from dotenv import load_dotenv
import os

REST_URL = "http://data.bioontology.org"

# Load API key from .env file
load_dotenv()
API_KEY = os.environ.get("BIO_PORTAL_API_KEY", "")

def get_json(url):
    opener = urllib.request.build_opener()
    opener.addheaders = [('Authorization', 'apikey token=' + API_KEY)]
    return json.loads(opener.open(url).read())

# Access the DOID ontology specifically
doid_acronym = "DOID"  # Human Disease Ontology
doid_url = f"{REST_URL}/ontologies/{doid_acronym}"
doid_ontology = get_json(doid_url)
print(f"Accessing DOID: {doid_ontology['name']}")

# Get the root classes of the DOID ontology
roots_url = doid_ontology['links']['roots']
roots = get_json(roots_url)
print("\nRoot classes in DOID:")
if len(roots) > 1:
    for root in roots:
        print(f"- {root['prefLabel']}")
        if root['prefLabel'] == 'symptom':
            symptom_root = root
            print("\tFOUND SYMPTOM ROOT: ", root)
        
elif len(roots) == 0: 
    print("No roots in DOID")
else: 
    print("\tSINGLE ROOT: ", root['prefLabel']) # type(root) == dict
    print()

    # The code checks if 'links' exists in root and if 'children' exists in root['links'], then prints the children link.
    # This is correct if you want to print the URL to the children of the root class in the ontology.
    if 'links' in root and 'children' in root['links']:
        pprint(root['links']['children'])





In [ ]:
# Symptom Root
symptom_root_url = symptom_root['links']['self']
print("SYMPTOM ROOT URL: ", symptom_root_url)
symptom_root_info = get_json(symptom_root_url)
print("definition of a symptom: ", symptom_root_info['definition'])
# Look at page with the children of of the root symptom
symptom_children = get_json(symptom_root_info['links']['children'])
symptom_children
# track the symptoms 
l1_symptoms = []
if len(symptom_children['collection'])> 0:
    print("Total number of symptoms: ", len(symptom_children['collection']))
    for symptom in symptom_children['collection']:
        print(" -> ", symptom)
        info = {
            'prefLabel': symptom['prefLabel'],
            'synonym': symptom['synonym'],
            'definition': symptom['definition'],
            'links': symptom['links'], 
        }
        l1_symptoms.append(info)


In [ ]:
level_1_symptoms = []
for symptom in children['collection']:
    print(symptom)
    info = {
        'prefLabel': symptom['prefLabel'],
        'synonym': symptom['synonym'],
        'definition': symptom['definition'],
        'links': symptom['links'], 
    }
    level_1_symptoms.append(info)
    # 'prefLabel', 'synonym', 'definition'
print(len(level_1_symptoms))

In [ ]:
l1_symptoms[0]['prefLabel']

In [ ]:
children_of_symptom_0 = l1_symptoms[0]['links']['children']
children_of_symptom_0 = get_json(children_of_symptom_0)
print("Number of children: ", len(children_of_symptom_0['collection']))
for child in children_of_symptom_0['collection']:
    print(f" --> {child['prefLabel']}")

In [ ]:
level_1_symptoms[0]['prefLabel']
#level_1_symptoms[0]['links']['children']
c1 = get_json(level_1_symptoms[0]['links']['children'])
c1

In [ ]:
def traverse_symptom_tree(node_url, level=0, max_depth=None, verbose=True):
    """
    Recursively traverse the symptom tree with max depth detection
    
    Args:
        node_url: URL of the current node
        level: Current depth level (for indentation)
        max_depth: Maximum depth to traverse (None for unlimited)
        verbose: Print traversal information
    
    Returns:
        dict: Node information with children, or None if max depth reached
    """
    # Check if we've reached max depth
    if max_depth is not None and level >= max_depth:
        if verbose:
            print(f"🛑 MAX DEPTH REACHED at level {level} (max_depth={max_depth})")
        return None
    
    try:
        # Get the current node
        node_data = get_json(node_url)
        
        if verbose:
            print(f"📁 Processing level {level}: {node_data.get('prefLabel', 'Unknown')}")
        
        # Extract essential information
        node_info = {
            'prefLabel': node_data.get('prefLabel', ''),
            'synonym': node_data.get('synonym', []),
            'definition': node_data.get('definition', []),
            'level': level,
            'children': [],
            'max_depth_reached': False  # Flag to track if max depth was hit
        }
        
        # Check if this node has children
        if 'collection' in node_data:
            print(" COLLECTION IN NODE DATA")

        if 'links' in node_data and 'children' in node_data['links']:
            node_info['links'] = node_data['links']
            children_url = node_data['links']['children']
            children_data = get_json(children_url)
        
            if verbose:
                print(f"  �� Found {len(children_data['collection'])} children at level {level}")
            
            # Recursively process each child
            for i, child in enumerate(children_data['collection']):
                child_url = child['links']['self']
                child_info = traverse_symptom_tree(child_url, level + 1, max_depth, verbose)
                
                if child_info is None:
                    # Max depth was reached for this child
                    node_info['max_depth_reached'] = True
                    if verbose:
                        print(f"  ⚠️  Child {i+1} stopped at max depth")
                else:
                    node_info['children'].append(child_info)
        
        return node_info
        
    except Exception as e:
        if verbose:
            print(f"❌ Error processing node at level {level}: {str(e)}")
        return None

def print_tree_with_depth_info(node, indent=0):
    """
    Print the tree with depth information and max depth indicators
    """
    if not node:
        return
    
    # Print current node with depth indicator
    prefix = "  " * indent
    depth_indicator = f"[L{node['level']}]"
    max_depth_flag = " 🛑" if node.get('max_depth_reached', False) else ""
    
    print(f"{prefix}{depth_indicator} → {node['prefLabel']}{max_depth_flag}")
    
    # Print children
    for child in node['children']:
        print_tree_with_depth_info(child, indent + 1)

# Usage with max depth detection
root_url =  "https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FSYMP_0000462"
# Traverse with max depth of 2
tree = traverse_symptom_tree(root_url, max_depth=None, verbose=True)
print("\n" + "="*50)
print("TREE STRUCTURE WITH DEPTH INFO:")
print_tree_with_depth_info(tree)

In [ ]:
import json

def save_tree_to_json(tree_data, file_path):
    """
    Save the symptom tree to a JSON file
    
    Args:
        tree_data: The tree data structure
        file_path: Path to save the JSON file
    """
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(tree_data, f, indent=2, ensure_ascii=False)
    
    print(f"Tree saved to {file_path}")

# Usage
save_tree_to_json(tree, "symptom_tree.json")

import csv

def tree_to_csv_rows(node, path="", rows=None):
    """Convert tree to flat CSV rows with path information"""
    if rows is None:
        rows = []
    
    if not node:
        return rows
    
    # Create current path
    current_path = f"{path}/{node['prefLabel']}" if path else node['prefLabel']
    
    # Add this node as a row
    row = {
        'path': current_path,
        'level': node['level'],
        'prefLabel': node['prefLabel'],
        'synonym': "|".join(node.get('synonym', [])),
        'definition': "|".join(node.get('definition', []))
    }
    rows.append(row)
    
    # Process children
    for child in node.get('children', []):
        tree_to_csv_rows(child, current_path, rows)
    
    return rows

def save_tree_to_csv(tree_data, file_path):
    """Save tree as a flat CSV with path information"""
    rows = tree_to_csv_rows(tree_data)
    
    if rows:
        fieldnames = rows[0].keys()
        with open(file_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        
        print(f"Tree saved to {file_path} ({len(rows)} rows)")

# Usage
save_tree_to_csv(tree, "symptom_tree.csv")


# import sqlite3

# def save_tree_to_sqlite(tree_data, db_path):
#     """Save tree to SQLite database for efficient querying"""
#     conn = sqlite3.connect(db_path)
#     cursor = conn.cursor()
    
#     # Create tables
#     cursor.execute('''
#     CREATE TABLE IF NOT EXISTS symptoms (
#         id INTEGER PRIMARY KEY AUTOINCREMENT,
#         prefLabel TEXT,
#         level INTEGER,
#         parent_id INTEGER,
#         FOREIGN KEY (parent_id) REFERENCES symptoms(id)
#     )
#     ''')
    
#     cursor.execute('''
#     CREATE TABLE IF NOT EXISTS symptom_metadata (
#         symptom_id INTEGER,
#         type TEXT,
#         value TEXT,
#         FOREIGN KEY (symptom_id) REFERENCES symptoms(id)
#     )
#     ''')
    
#     # Helper function to insert nodes recursively
#     def insert_node(node, parent_id=None):
#         if not node:
#             return
        
#         # Insert the node
#         cursor.execute(
#             "INSERT INTO symptoms (prefLabel, level, parent_id) VALUES (?, ?, ?)",
#             (node['prefLabel'], node['level'], parent_id)
#         )
#         node_id = cursor.lastrowid
        
#         # Insert synonyms
#         for synonym in node.get('synonym', []):
#             cursor.execute(
#                 "INSERT INTO symptom_metadata (symptom_id, type, value) VALUES (?, ?, ?)",
#                 (node_id, 'synonym', synonym)
#             )
        
#         # Insert definitions
#         for definition in node.get('definition', []):
#             cursor.execute(
#                 "INSERT INTO symptom_metadata (symptom_id, type, value) VALUES (?, ?, ?)",
#                 (node_id, 'definition', definition)
#             )
        
#         # Process children
#         for child in node.get('children', []):
#             insert_node(child, node_id)
    
#     # Start inserting from root
#     insert_node(tree_data)
#     conn.commit()
#     conn.close()
    
#     print(f"Tree saved to SQLite database: {db_path}")

# # Usage
# save_tree_to_sqlite(tree, "symptom_tree.db")

In [ ]:
def print_tree_like_website(node, level=0, is_last=True, prefix=""):
    """
    Print the tree structure exactly like the BioPortal website
    Uses proper indentation and tree-like visual indicators
    """
    if not node:
        return
    
    # Determine the tree connector symbols
    if level == 0:
        # Root level
        connector = "▼ "
        print(f"{connector}{node['prefLabel']}")
    else:
        # Child levels - use proper tree structure
        if is_last:
            connector = "└─ "
            new_prefix = prefix + "   "
        else:
            connector = "├─ "
            new_prefix = prefix + "│  "
        
        print(f"{prefix}{connector}{node['prefLabel']}")
        prefix = new_prefix
    
    # Print children
    children = node.get('children', [])
    for i, child in enumerate(children):
        is_last_child = (i == len(children) - 1)
        print_tree_like_website(child, level + 1, is_last_child, prefix if level > 0 else "")

def print_expandable_tree(node, level=0, expanded_levels=None):
    """
    Print tree with expandable/collapsible indicators like the website
    
    Args:
        node: The tree node
        level: Current level
        expanded_levels: Set of levels to show as expanded (None = all expanded)
    """
    if not node:
        return
    
    if expanded_levels is None:
        expanded_levels = set(range(10))  # Expand all levels by default
    
    # Indentation
    indent = "  " * level
    
    # Determine expand/collapse indicator
    has_children = len(node.get('children', [])) > 0
    
    if level == 0:
        # Root level
        if has_children:
            indicator = "▼ " if level in expanded_levels else "▶ "
        else:
            indicator = ""
        print(f"{indicator}{node['prefLabel']}")
    else:
        # Child levels
        if has_children:
            indicator = "▼ " if level in expanded_levels else "▶ "
        else:
            indicator = ""
        print(f"{indent}{indicator}{node['prefLabel']}")
    
    # Print children only if this level is expanded
    if level in expanded_levels:
        for child in node.get('children', []):
            print_expandable_tree(child, level + 1, expanded_levels)

def print_website_style(node, level=0):
    """
    Most accurate reproduction of the website's tree style
    """
    if not node:
        return
    
    # Calculate indentation
    indent = "  " * level
    
    # Determine the appropriate symbol
    children = node.get('children', [])
    has_children = len(children) > 0
    
    if level == 0:
        # Root level with dropdown arrow
        symbol = "▼ " if has_children else ""
        print(f"{symbol}{node['prefLabel']}")
    else:
        # Child levels
        if has_children:
            # Has children - show dropdown arrow
            symbol = "▼ "
        else:
            # No children - show right arrow
            symbol = "▶ "
        
        print(f"{indent}{symbol}{node['prefLabel']}")
    
    # Print all children
    for child in children:
        print_website_style(child, level + 1)

def print_compact_tree(node, level=0, max_display_level=3):
    """
    Compact version that limits display depth for readability
    """
    if not node or level > max_display_level:
        return
    
    indent = "  " * level
    children = node.get('children', [])
    
    if level == 0:
        print(f"▼ {node['prefLabel']}")
    else:
        symbol = "▼" if children and level < max_display_level else "▶"
        print(f"{indent}{symbol} {node['prefLabel']}")
        
        # Show count if there are hidden children
        if children and level >= max_display_level:
            print(f"{indent}  ... ({len(children)} subcategories)")
    
    # Only print children if we haven't reached max display level
    if level < max_display_level:
        for child in children:
            print_compact_tree(child, level + 1, max_display_level)

# Usage examples:

print("=== WEBSITE STYLE (Most Accurate) ===")
print_website_style(tree)

# print("\n=== TREE STRUCTURE WITH CONNECTORS ===")
# print_tree_like_website(tree)

# print("\n=== EXPANDABLE STYLE ===")
# # Show only first 2 levels expanded
# expanded_levels = {0, 1}
# print_expandable_tree(tree, expanded_levels=expanded_levels)

# print("\n=== COMPACT VIEW ===")
# print_compact_tree(tree, max_display_level=2)